# Hướng dẫn train baseline Time-Series (từng bước)

Notebook này hướng dẫn **chạy từng cell** để:
1. Load dữ liệu `sales.csv` và `sample_submission.csv`
2. Chạy **Time-Series Cross Validation** (expanding window)
3. Train baseline **Linear Regression** trên toàn bộ train
4. Tạo file `submission.csv` và `baseline_metrics.json` để test pipeline

## Bước 0 - Lưu ý trước khi chạy

- Chạy từ thư mục `notebooks/`.
- Nếu chưa có môi trường: mở terminal và chạy `uv sync`.
- Nếu đã chạy `src/forecasting.py` trước đó, file output cũ sẽ được ghi đè.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Chạy cell này để xác định đường dẫn dữ liệu/output
ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

sales_path = (PROCESSED_DIR / 'sales.csv') if (PROCESSED_DIR / 'sales.csv').exists() else (RAW_DIR / 'sales.csv')
submission_template_path = (
    (PROCESSED_DIR / 'sample_submission.csv')
    if (PROCESSED_DIR / 'sample_submission.csv').exists()
    else (RAW_DIR / 'sample_submission.csv')
)

print('ROOT              :', ROOT)
print('sales_path        :', sales_path)
print('template_path     :', submission_template_path)
print('output_submission :', PROCESSED_DIR / 'submission.csv')
print('output_metrics    :', PROCESSED_DIR / 'baseline_metrics.json')

In [ ]:
# Chạy cell này để load và làm sạch dữ liệu cơ bản
sales_df = pd.read_csv(sales_path, low_memory=False)
template_df = pd.read_csv(submission_template_path, low_memory=False)

sales_df['Date'] = pd.to_datetime(sales_df['Date'], errors='coerce')
sales_df['Revenue'] = pd.to_numeric(sales_df['Revenue'], errors='coerce')
sales_df = sales_df.dropna(subset=['Date', 'Revenue']).sort_values('Date').drop_duplicates('Date', keep='last').reset_index(drop=True)

template_df['Date'] = pd.to_datetime(template_df['Date'], errors='coerce')
if template_df['Date'].isna().any():
    raise ValueError('sample_submission.csv co Date khong hop le.')

print('sales_df shape    :', sales_df.shape)
print('template_df shape :', template_df.shape)
display(sales_df.head(3))

In [ ]:
# Chay cell này để khai báo hàm tạo feature và đánh giá
def date_features(dates: pd.Series, origin: pd.Timestamp) -> pd.DataFrame:
    days_since = (dates - origin).dt.days.astype(float)
    day_of_week = dates.dt.dayofweek.astype(float)
    month = dates.dt.month.astype(float)
    day_of_year = dates.dt.dayofyear.astype(float)
    week_sin = np.sin(2.0 * np.pi * day_of_week / 7.0)
    week_cos = np.cos(2.0 * np.pi * day_of_week / 7.0)
    year_sin = np.sin(2.0 * np.pi * day_of_year / 365.25)
    year_cos = np.cos(2.0 * np.pi * day_of_year / 365.25)
    return pd.DataFrame(
        {
            'days_since': days_since,
            'day_of_week': day_of_week,
            'month': month,
            'week_sin': week_sin,
            'week_cos': week_cos,
            'year_sin': year_sin,
            'year_cos': year_cos,
        }
    )

def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2': float(r2_score(y_true, y_pred)),
    }

In [ ]:
# Chạy cell này để thiết lập Time-Series CV (expanding window)
CV_FOLDS = 5
CV_VALID_SIZE = 30
CV_MIN_TRAIN_SIZE = 180

required_rows = CV_MIN_TRAIN_SIZE + CV_FOLDS * CV_VALID_SIZE
if len(sales_df) < required_rows:
    raise ValueError(
        f'Khong du du lieu cho CV. Can it nhat {required_rows} dong, hien tai co {len(sales_df)} dong.'
    )

fold_metrics = []
for fold in range(CV_FOLDS):
    train_end = CV_MIN_TRAIN_SIZE + fold * CV_VALID_SIZE
    valid_start = train_end
    valid_end = valid_start + CV_VALID_SIZE

    train_df = sales_df.iloc[:train_end].copy()
    valid_df = sales_df.iloc[valid_start:valid_end].copy()

    origin = train_df['Date'].min()
    x_train = date_features(train_df['Date'], origin)
    y_train = train_df['Revenue'].to_numpy()
    x_valid = date_features(valid_df['Date'], origin)
    y_valid = valid_df['Revenue'].to_numpy()

    model = LinearRegression()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_valid)

    metrics = evaluate(y_valid, y_pred)
    fold_metrics.append({
        'fold': fold + 1,
        'train_rows': int(train_df.shape[0]),
        'valid_rows': int(valid_df.shape[0]),
        **metrics,
    })

cv_df = pd.DataFrame(fold_metrics)
display(cv_df)
print('\nCV trung binh:')
print(cv_df[['mae', 'rmse', 'r2']].mean().to_dict())

In [ ]:
# Chạy cell này để train trên toàn bộ sales_df và dự báo cho tập test (template)
origin_full = sales_df['Date'].min()
x_full = date_features(sales_df['Date'], origin_full)
y_full = sales_df['Revenue'].to_numpy()
x_test = date_features(template_df['Date'], origin_full)

final_model = LinearRegression()
final_model.fit(x_full, y_full)
revenue_pred = np.maximum(final_model.predict(x_test), 0.0)

submission_df = template_df.copy()
submission_df['Revenue'] = revenue_pred

if 'COGS' not in submission_df.columns:
    if 'COGS' in sales_df.columns:
        ratio = float((sales_df['COGS'] / sales_df['Revenue']).replace([np.inf, -np.inf], np.nan).dropna().mean())
        if np.isnan(ratio):
            ratio = 0.1
    else:
        ratio = 0.1
    submission_df['COGS'] = submission_df['Revenue'] * ratio

display(submission_df.head(5))

In [ ]:
# Chạy cell này để lưu output pipeline
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

submission_out = PROCESSED_DIR / 'submission.csv'
metrics_out = PROCESSED_DIR / 'baseline_metrics.json'

submission_df.to_csv(submission_out, index=False)

payload = {
    'selected_method': 'linear_regression',
    'selected_metrics': cv_df[['mae', 'rmse', 'r2']].mean().to_dict(),
    'cv': {
        'strategy': 'expanding_window',
        'config': {
            'folds': CV_FOLDS,
            'valid_size': CV_VALID_SIZE,
            'min_train_size': CV_MIN_TRAIN_SIZE,
        },
        'fold_metrics': fold_metrics,
    },
    'submission_path': str(submission_out),
}

metrics_out.write_text(json.dumps(payload, indent=2), encoding='utf-8')

print('Da luu:', submission_out)
print('Da luu:', metrics_out)

## Lệnh CLI tương đương (nếu muốn chạy nhanh không qua notebook)

```bash
uv run src/forecasting.py --method linear_regression --cv-folds 5 --cv-valid-size 30 --cv-min-train-size 180
```